*<small>Last updated: 2026-03-08 23:42:48 UTC | Student Version (No Solutions)</small>*

# Character-level GPT with Hugging Face Transformers and PyTorch Lightning

In this notebook we build a small **generative language model** using the GPT architecture.
Instead of writing every layer from scratch, we use the [Hugging Face `transformers`](https://huggingface.co/docs/transformers) library, which packages state-of-the-art architectures in a consistent, well-tested API. We wrap training with **PyTorch Lightning** to keep the training loop clean and reproducible.

**What you will learn:**
- How next-token prediction is framed as a supervised learning problem
- What a decoder-only transformer is and how it differs from encoder / encoder-decoder architectures
- Where token embeddings, positional embeddings, and the causal attention mask live inside an HF model
- How Hugging Face handles label shifting automatically
- How autoregressive text generation works step by step
- How to visualise attention scores as a heatmap

**Prerequisites:** familiarity with PyTorch (`nn.Module`, `Dataset`, `DataLoader`) and PyTorch Lightning (`LightningModule`, `LightningDataModule`, `Trainer`). Hugging Face is introduced here from scratch.

---
## Section 1 — Setup

In [ ]:
%pip install transformers lightning matplotlib seaborn --quiet

In [ ]:
import urllib.request

import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import DataLoader, Dataset

import lightning as L
from transformers import GPT2Config, GPT2LMHeadModel

# Reproducibility
torch.manual_seed(42)

---
## Section 2 — Data: character-level language modelling

### The task

We train our model on a large text file. The goal is simple: **given any sequence of characters, predict the next character**. This is called *causal* or *autoregressive* language modelling.

The word *causal* means the model is only allowed to look at characters that appear *before* the current position — never into the future. This constraint is what makes the model useful for generation: after training, we can feed it a prompt and let it extend the text one character at a time.

### The dataset: Tiny Shakespeare

We use a concatenation of Shakespeare's works (~1 million characters). It is small enough to train on a laptop in a few minutes, yet rich enough to produce recognisable Shakespearean text.

In [ ]:
# ── Download the dataset ─────────────────────────────────────────────────────
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "input.txt")

with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(f"Total characters in dataset: {len(text):,}")
print("First 200 characters:")
print(text[:200])

In [ ]:
# ── Character-level tokenizer ─────────────────────────────────────────────────
#
# Tokenization converts raw text into integers that the model can process.
# Here we use the simplest possible tokenizer: one integer per unique character.
#
# More powerful models (GPT-4, LLaMA, etc.) use sub-word tokenizers (BPE) that
# assign tokens to frequent character groups, shrinking sequence lengths and
# allowing much larger vocabularies. Character-level keeps things transparent.

chars = sorted(set(text))          # all unique characters, sorted for reproducibility
vocab_size = len(chars)
print(f"Vocabulary size: {vocab_size} unique characters")
print(f"Characters: {''.join(chars)!r}")

# stoi: string → integer  |  itos: integer → string
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s: str) -> list[int]:
    """Convert a string to a list of integer token IDs."""
    return [stoi[c] for c in s]

def decode(ids: list[int]) -> str:
    """Convert a list of integer token IDs back to a string."""
    return "".join(itos[i] for i in ids)

# Encode the whole dataset once
data_encoded = encode(text)
print(f"\nEncoded length: {len(data_encoded):,} tokens")
print(f"First 20 tokens: {data_encoded[:20]}")
print(f"Decoded back: {decode(data_encoded[:20])!r}")

### How next-token prediction works — the supervised learning view

This is the most important concept to grasp before looking at any model code.

We slide a window of `block_size` tokens over the text. For each window:
- The **input** `x` is the window itself: tokens at positions `[0, 1, …, T-1]`
- The **target** `y` is the window shifted **one step to the right**: tokens at positions `[1, 2, …, T]`

At every position `t` inside the window, the model sees tokens `[0 … t]` and must predict token `t+1`. One window therefore gives us `block_size` training examples simultaneously — very efficient.

Let us print this out concretely:

In [ ]:
# ── Visualise next-token prediction ──────────────────────────────────────────
sample_text     = "To be, or not to be, that is the"
sample_ids      = encode(sample_text)
block_size_demo = 10  # smaller than real block_size so the table fits on screen

x_demo = sample_ids[:block_size_demo]       # input window
y_demo = sample_ids[1:block_size_demo + 1]  # target = x shifted right by 1

print(f"Input  x : {decode(x_demo)!r}")
print(f"Target y : {decode(y_demo)!r}")
print()
print("Position-by-position view (one training example per row):")
print(f"  {'Context seen by the model':32s}  →  Target (next character)")
print(f"  {'-'*58}")
for t in range(block_size_demo):
    context  = decode(x_demo[: t + 1])
    next_tok = decode([y_demo[t]])
    print(f"  {context!r:32s}  →  {next_tok!r}")

Notice that `y` is simply `x` shifted one position to the right — `y[t] = x[t+1]`.

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
BLOCK_SIZE  = 32   # context length: how many characters the model sees at once
BATCH_SIZE  = 16
MAX_STEPS    = 50000
LEARNING_RATE = 1e-3

N_EMBD  = 64   # embedding / hidden dimension, d_model
N_LAYER = 4    # number of transformer blocks (depth)
N_HEAD  = 4    # attention heads per block (width of attention)

In [ ]:
# ── Dataset ───────────────────────────────────────────────────────────────────
class CharDataset(Dataset):
    """
    Slides a window of `block_size` tokens over the encoded text.
    Returns a single tensor x of shape [block_size].
    The model will use x as both input_ids AND labels (HF shifts internally).
    """
    def __init__(self, data: list[int], block_size: int):
        self.data       = data
        self.block_size = block_size

    def __len__(self):
        # Every valid starting position yields one sample
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        window = self.data[idx : idx + self.block_size]
        return torch.tensor(window, dtype=torch.long)

In [ ]:
# ── DataModule (80 / 10 / 10 split) ──────────────────────────────────────────
class ShakespeareDataModule(L.LightningDataModule):
    """
    Wraps three DataLoaders (train / val / test) in a Lightning DataModule.
    Splitting the data into three non-overlapping parts allows us to:
      - train on the training set
      - tune / monitor on the validation set without touching the test set
      - report final unbiased performance on the test set
    """
    def __init__(self, data: list[int], block_size: int, batch_size: int):
        super().__init__()
        self.data       = data
        self.block_size = block_size
        self.batch_size = batch_size

    def setup(self, stage=None):
        n          = len(self.data)
        train_data = self.data[: int(0.8 * n)]
        val_data   = self.data[int(0.8 * n) : int(0.9 * n)]
        test_data  = self.data[int(0.9 * n) :]

        self.train_ds = CharDataset(train_data, self.block_size)
        self.val_ds   = CharDataset(val_data,   self.block_size)
        self.test_ds  = CharDataset(test_data,  self.block_size)

        print(f"Dataset sizes  —  train: {len(self.train_ds):,}  "
              f"| val: {len(self.val_ds):,}  "
              f"| test: {len(self.test_ds):,}  (samples)")

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.val_ds, batch_size=self.batch_size)

    def test_dataloader(self):
        return DataLoader(self.test_ds, batch_size=self.batch_size)

dm     = ShakespeareDataModule(data_encoded, block_size=BLOCK_SIZE, batch_size=BATCH_SIZE)
dm.setup()  # prepare the datasets and dataloaders


In [ ]:
# ── GPT-2 configuration ───────────────────────────────────────────────────────
#
# GPT2Config is a plain dataclass that holds every architectural hyperparameter.
# GPT2LMHeadModel builds the full model from it:
#   - token embedding + positional embedding
#   - n_layer transformer blocks (each with multi-head attention + MLP)
#   - final layer normalisation
#   - language-modelling head (linear projection to vocab_size logits)

config = GPT2Config(
    vocab_size  = vocab_size,  # 65 unique characters
    n_positions = BLOCK_SIZE,  # maximum sequence length the model can handle
    n_embd      = N_EMBD,      # embedding / hidden dimension  (64)
    n_layer     = N_LAYER,     # number of stacked transformer blocks  (4)
    n_head      = N_HEAD,      # attention heads per block  (4)
    # each head attends over n_embd // n_head = 64 // 4 = 16 dimensions
    resid_pdrop = 0.0,         # residual dropout (disabled — simple baseline)
    embd_pdrop  = 0.0,
    attn_pdrop  = 0.0,
)

model_hf = GPT2LMHeadModel(config)
n_params  = sum(p.numel() for p in model_hf.parameters())
print(f"Total trainable parameters: {n_params:,}")

In [ ]:
import torch.nn.functional as F

x = next(iter(dm.train_dataloader()))[:3]  # small batch for speed
print(x)

# HF computes loss internally
out_with_labels = model_hf(input_ids=x, labels=x)
hf_loss = out_with_labels.loss

# Get raw logits (no labels → no internal loss computation)
out_logits_only = model_hf(input_ids=x)
logits = out_logits_only.logits  # [B, T, V]
print("Logits shape", logits.shape)

# Manually replicate HF's internal shift
#logits[:, :-1, :]   # [3, 31, 65]  — drop last position's prediction
#logits[:, :-1, :].reshape(-1, logits.size(-1)) # [3, 31, 65] → [3*31, 65] = [93, 65] #   -1 inferred as 3×31=93

#x[:, 1:]            # [3, 31]      — drop first token (nothing predicts it)
# x[:, 1:].reshape(-1) [3, 31] → [3*31] = [93] #   -1 inferred as 3×31=93

manual_loss = F.cross_entropy(
    logits[:, :-1, :].reshape(-1, logits.size(-1)),
    x[:, 1:].reshape(-1)
)

print(f"HF loss:     {hf_loss.item():.6f}")
print(f"Manual loss: {manual_loss.item():.6f}")
print(f"Match: {torch.allclose(hf_loss, manual_loss)}")



#### How Hugging Face handles the shift automatically

In a manual training loop you would build `(x, y)` pairs explicitly as above and compute
`cross_entropy(logits, y)` yourself.

Hugging Face does something equivalent internally: when you call
```python
model(input_ids=x, labels=x)
```
it automatically computes
```python
loss = cross_entropy(logits[:, :-1, :], labels[:, 1:])
```
that is, it shifts the labels by one position before computing the loss.

**This means you do not need a separate `y` tensor at all.** You pass the same sequence as both `input_ids` and `labels`. The `CharDataset` below therefore returns only `x`.

> ⚠️ **Common mistake:** if you pass a *pre-shifted* `y` as `labels`, HF shifts it again internally, and you end up training the model to predict two tokens ahead — a silent bug that still compiles and runs but produces a worse model.

---
## Section 3 — Architecture

### Decoder-only transformers and their siblings

The transformer architecture comes in three main flavours, each suited to a different class of tasks:

| Architecture | Attention style | Example models | Typical applications |
|---|---|---|---|
| **Encoder-only** | Bidirectional (sees full sequence) | BERT, RoBERTa | Text classification, named-entity recognition, sentence embeddings |
| **Encoder-decoder** | Encoder: bidirectional · Decoder: causal | T5, BART, mT5 | Machine translation, summarisation, question answering |
| **Decoder-only** | Causal (left-to-right only) | GPT-2, GPT-4, LLaMA | Text generation, language modelling, chat |

We are building a **decoder-only** model. Its defining property is the **causal attention mask**: every token can only attend to tokens that come *before* it (or itself). This prevents the model from "peeking at the answer" during training, and is the mechanism that makes autoregressive generation possible at inference time.

An encoder-only model allows bidirectional attention — every token attends to every other token simultaneously. This is powerful when you have the full input available (e.g. for classification), but it cannot be used to generate one token at a time because it would need to see the tokens it has not yet generated.

---

### `n_layer` vs `n_head` — depth vs width of attention

**`n_layer` — how many transformer blocks are stacked (depth)**

Each transformer block is a complete processing stage: multi-head self-attention → feedforward network, both with residual connections and layer normalisation. Stacking `n_layer` blocks means the signal passes through `n_layer` rounds of refinement. Intuitively, early blocks capture local, surface patterns (nearby characters); later blocks build more abstract, long-range structure. More layers → more expressive model, but also more parameters and slower training.

**`n_head` — how many parallel attention views each block has (width)**

Within a single block, multi-head attention splits the embedding dimension into `n_head` independent subspaces. Each head runs its own scaled dot-product attention over the full sequence, potentially learning to focus on a different linguistic aspect (e.g. one head might track syntactic structure, another might track repetition). Their outputs are concatenated and projected back to the original dimension. Each head operates on `n_embd / n_head` dimensions.

**Analogy:** `n_layer` is how many times you re-read a sentence; `n_head` is how many different things you pay attention to on each read.

In [ ]:
# ── Print the full module tree ────────────────────────────────────────────────
#
# This shows the entire architecture as PyTorch sees it.
# Key things to spot:
#   transformer.wte  — token embedding table
#   transformer.wpe  — positional embedding table
#   transformer.h[0..3] — the four transformer blocks; inside each:
#       ln_1, attn  — layer norm + multi-head self-attention
#       ln_2, mlp   — layer norm + feedforward network
#   transformer.ln_f — final layer norm
#   lm_head          — linear layer projecting to vocab_size logits

print(model_hf)
print()
print(f"{'Module':52s}  {'Parameters':>12}")
print("-" * 68)
for name, module in model_hf.named_modules():
    params = sum(p.numel() for p in module.parameters(recurse=False))
    if params > 0:
        print(f"  {name:50s}  {params:>12,}")



### Why `Conv1D` instead of `nn.Linear`?

HF's `Conv1D` is **not** PyTorch's `nn.Conv1d`. It is a custom class defined in HuggingFace
transformers that is **mathematically identical to `nn.Linear`**, just with the weight matrix
stored transposed.

| | Weight shape | Computation |
|---|---|---|
| `nn.Linear(64, 192)` | `[192, 64]` | `x @ W.T + b` |
| `Conv1D(nx=64, nf=192)` | `[64, 192]` | `x @ W + b` (no transpose) |

Same result, different memory layout.

#### Why the name "Conv1D"?

It comes from the **original OpenAI GPT-2 TensorFlow implementation**. A 1D convolution with
`kernel_size=1` applied over a sequence is exactly a linear layer applied independently at each
position — it "slides" a `[1, in, out]` kernel over the sequence, which degenerates to a matrix
multiply. HuggingFace preserved this naming for weight compatibility when loading the original
pretrained GPT-2 checkpoints.

#### Why does it matter?

It doesn't affect the math at all. `c_attn: Conv1D(nf=192, nx=64)` produces exactly the same
output as `nn.Linear(64, 192)` would. The only practical difference is the weight checkpoint
format — if you load OpenAI's original weights, they fit directly without transposing.


### Where are the 4 heads? — reading the `forward()` source

The module tree above shows **no submodule per head**. That is intentional: HF fuses all heads into a single `c_attn` layer and creates them at runtime with a tensor reshape inside `forward()`.

Use `inspect.getsource` to read the actual implementation.

In [ ]:
# ── Print the GPT2Attention.forward() source ──────────────────────────────────
#
# The 4 attention heads are NOT separate nn.Module objects.
# They are created inside forward() via two tensor operations:
#
#   1. self.c_attn(hidden_states).split(self.split_size, dim=2)
#      → splits the 192-dim output into Q, K, V  (each [batch, seq, 64])
#
#   2. query.view(*query.shape[:-1], -1, self.head_dim).transpose(1, 2)
#      → reshapes [batch, seq, 64]  →  [batch, 4, seq, 16]
#                                               ↑
#                                         the 4 heads appear here
#
# head_dim = n_embd // n_head = 64 // 4 = 16

import inspect
from transformers.models.gpt2.modeling_gpt2 import GPT2Attention

print(inspect.getsource(GPT2Attention.forward))

### Token embeddings and positional embeddings

The transformer operates on continuous vectors, not raw integers. Two learned embedding tables convert the input:

- **Token embedding (`wte`)** — an `nn.Embedding` of shape `[vocab_size, n_embd]`. Each row is the learned representation of one character. When we look up token ID `i`, we retrieve row `i`.

- **Positional embedding (`wpe`)** — an `nn.Embedding` of shape `[block_size, n_embd]`. Each row is the learned representation of one *position* (0, 1, 2, …, block_size−1). Unlike the sinusoidal encoding in the original "Attention is All You Need" paper, GPT-2 learns these embeddings end-to-end from data.

The two embeddings are **summed** to form the input to the transformer blocks. This means the model's input carries both *what* the token is and *where* it appears in the sequence.

In [ ]:
# ── Inspect embedding tables ──────────────────────────────────────────────────
wte = model_hf.transformer.wte   # Token Embedding
wpe = model_hf.transformer.wpe   # Positional Embedding

print("Token embedding     (wte) shape:", wte.weight.shape)   # [vocab_size, n_embd]
print("Positional embedding(wpe) shape:", wpe.weight.shape)   # [block_size, n_embd]

# Manually trace what happens to a small batch
dummy_ids = torch.randint(0, vocab_size, (1, 10))   # batch=1, seq_len=10
positions = torch.arange(10).unsqueeze(0)            # [[0, 1, 2, …, 9]]

tok_emb = wte(dummy_ids)      # [1, 10, 64]
pos_emb = wpe(positions)      # [1, 10, 64]
x_input = tok_emb + pos_emb   # [1, 10, 64]  ← what enters the transformer blocks

print(f"\ntok_emb shape: {tok_emb.shape}")
print(f"pos_emb shape: {pos_emb.shape}")
print(f"x_input shape: {x_input.shape}  (tok_emb + pos_emb)")

# Verify that lm_head shares weights with wte (tied embeddings)
"""
 Tied embeddings.

wte (Weight Token Embeddings): This is the embedding layer that converts input token IDs (integers) into dense vector representations.
Essentially, it's a lookup table where each row corresponds to a unique token in your vocabulary, and the columns are the embedding dimensions.

lm_head (Language Model Head): This is the final linear layer of the model that takes the contextualized token representations from the Transformer blocks
and projects them back into the size of the vocabulary. Its output is the logits (raw prediction scores) for each possible next token in the vocabulary.

Why are their weights tied?

Parameter Efficiency: Both wte and lm_head effectively perform similar operations but in reverse.
wte maps from a discrete token ID to a continuous embedding space, and lm_head maps from the continuous embedding space back to logits for discrete token IDs.
If the input embedding matrix W_embed is used, then the output projection matrix W_project can be its transpose (W_embed^T).
Tying these weights means you don't need to learn two separate large matrices, significantly reducing the total number of parameters in the model.

Regularization: Tying the weights acts as a form of regularization. It constrains the model by forcing these two layers to learn a more coherent representation,
preventing overfitting and potentially leading to better generalization, especially with smaller datasets.

Symmetry: There's a theoretical symmetry argument. If a word embedding is good at representing a word, then its transpose should be good at predicting that word.
By tying the weights, the model is encouraged to learn embeddings that are useful both for understanding input and generating output.

"""
tied = model_hf.lm_head.weight is model_hf.transformer.wte.weight
print(f"\nlm_head.weight is wte.weight (tied embeddings): {tied}")
print("  → The same matrix maps characters to vectors at the input"
      " AND scores vocabulary items at the output.")

### The causal attention mask

The causal mask enforces the left-to-right constraint: at position `i`, the model can attend to positions `0 … i` but **not** `i+1 … T−1` (the future). Without this mask, the model could trivially copy the answer from the next position during training and would learn nothing useful.

Mechanically, the mask is a lower-triangular boolean matrix:
- `1` (blue below) → this pair of positions is **allowed** to interact
- `0` (white below) → this pair is **blocked** (set to −∞ before the softmax, effectively zeroing the attention weight)

HF GPT-2 stores this mask as a **constant buffer** (`bias`) inside every attention block and applies it automatically during every forward pass. You never need to pass it manually.

In [ ]:
# ── Visualise the causal mask ─────────────────────────────────────────────────
# The mask lives in the first transformer block's attention module.
# Shape: [1, 1, n_positions, n_positions]
causal_mask = model_hf.transformer.h[0].attn.bias

T    = config.n_positions
mask = causal_mask[0, 0, :T, :T].float().numpy()   # 1 = allowed, 0 = masked
print(mask)
plt.figure(figsize=(5, 4))
sns.heatmap(
    mask, cmap="Blues", cbar=False,
    xticklabels=False, yticklabels=False, linewidths=0,
)
plt.title(
    "Causal (lower-triangular) attention mask\n"
    "Blue = allowed   White = blocked (future)"
)
plt.xlabel("Key position  (token being attended TO)")
plt.ylabel("Query position  (current token)")
plt.tight_layout()
plt.show()

### What Hugging Face handles for you — three subtle points

When moving from a hand-written model to an HF model, three things are done differently:

1. **GELU instead of ReLU in the feedforward blocks**
   GPT-2 uses the Gaussian Error Linear Unit (GELU) activation. Both GELU and ReLU are non-linear activations; GELU is smoother near zero and has been found to work slightly better in language models. For a small model like ours the practical difference is negligible.

2. **Tied token embeddings**
   HF GPT-2 reuses the token embedding matrix (`wte`) as the weight matrix of the final language-modelling head (`lm_head`). This means the same learned character vectors are used at the input (to look up tokens) **and** at the output (to score vocabulary items). This reduces the parameter count and often improves generalisation. We verified this above: `model_hf.lm_head.weight is model_hf.transformer.wte.weight → True`.

3. **Automatic label shifting**
   Calling `model(input_ids=x, labels=x)` triggers HF's internal loss computation:
   ```python
   shift_logits = logits[:, :-1, :]   # drop last position's prediction
   shift_labels = labels[:, 1:]       # drop first label (nothing predicts it)
   loss = cross_entropy(shift_logits, shift_labels)
   ```
   You pass the raw sequence once; HF aligns inputs and targets internally.

---
## Section 4 — Training

In [ ]:
# ── Lightning module ──────────────────────────────────────────────────────────
class GPTLightning(L.LightningModule):
    """
    Wraps GPT2LMHeadModel in a LightningModule.

    The three step methods (training / validation / test) are identical in
    structure: forward pass, log the loss. Lightning calls them at the right
    times and handles gradient accumulation, device placement, and logging.
    """

    def __init__(self, config: GPT2Config):
        super().__init__()
        self.model = GPT2LMHeadModel(config)

    def forward(self, input_ids: torch.Tensor):
        # Pass the same tensor as input_ids AND labels.
        # HF shifts labels by 1 internally — no manual (x, y) pairing needed.
        return self.model(input_ids=input_ids, labels=input_ids)

    def training_step(self, batch: torch.Tensor, batch_idx: int):
        out = self(batch)
        # on_step=True logs every step; flushed to CSV every log_every_n_steps
        self.log("train_loss", out.loss, prog_bar=True, on_step=True, on_epoch=False)
        return out.loss

    def validation_step(self, batch: torch.Tensor, batch_idx: int):
        out = self(batch)
        # on_epoch=True averages the loss over all validation batches
        self.log("val_loss", out.loss, prog_bar=True, on_epoch=True)

    def test_step(self, batch: torch.Tensor, batch_idx: int):
        out = self(batch)
        self.log("test_loss", out.loss, prog_bar=True, on_epoch=True)

    def configure_optimizers(self):
        # AdamW is the standard optimiser for transformer language models.
        # It is Adam with decoupled weight decay regularisation.
        return torch.optim.AdamW(self.parameters(), lr=LEARNING_RATE)

gpt_pl = GPTLightning(config)

In [ ]:
import time

class SimpleEpochProgressCallback(L.Callback):
    """Prints a one-line summary every 100 steps (after each validation run)."""

    def __init__(self):
        super().__init__()
        self.last_print_time = None

    def on_train_start(self, trainer, pl_module):
        """Print training configuration and record the overall start time."""
        self.last_print_time = time.time()

        train_dataloader = trainer.train_dataloader
        if hasattr(train_dataloader, 'batch_size'):
            batch_size = train_dataloader.batch_size
        elif hasattr(train_dataloader, 'loaders') and hasattr(train_dataloader.loaders, 'batch_size'):
            batch_size = train_dataloader.loaders.batch_size
        else:
            batch_size = getattr(train_dataloader, 'batch_size', 'N/A')

        num_batches = len(train_dataloader)
        print(f"\nTraining Configuration: Batch Size={batch_size}, "
              f"Steps per Epoch={num_batches}, Max Steps={trainer.max_steps}\n")

    def on_validation_epoch_end(self, trainer, pl_module):
        """Print step summary after each validation run (skips the initial sanity check)."""
        if trainer.sanity_checking:
            return

        now        = time.time()
        delta      = now - self.last_print_time if self.last_print_time else 0
        self.last_print_time = now

        step       = trainer.global_step
        train_loss = trainer.callback_metrics.get('train_loss', None)
        val_loss   = trainer.callback_metrics.get('val_loss',   None)

        parts = [f"Step {step}/{trainer.max_steps}"]
        if train_loss is not None:
            parts.append(f"Train Loss: {train_loss:.4f}")
        if val_loss is not None:
            parts.append(f"Val Loss: {val_loss:.4f}")
        parts.append(f"Time/100 steps: {delta:.2f}s")

        print(" | ".join(parts))

epoch_progress = SimpleEpochProgressCallback()

In [ ]:
# ── Instantiate model and data ────────────────────────────────────────────────
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger



checkpoint_cb = ModelCheckpoint(monitor="val_loss", mode="min", save_top_k=1)
csv_logger    = CSVLogger(save_dir=".")

# ── Trainer ───────────────────────────────────────────────────────────────────
trainer = L.Trainer(
    max_steps          = MAX_STEPS,
    val_check_interval = 100,
    callbacks          = [epoch_progress, checkpoint_cb],
    logger             = csv_logger,
    log_every_n_steps  = 50,
    enable_progress_bar= False,
)

trainer.fit(gpt_pl, datamodule=dm)

---
## Section 5 — Evaluation

After training we compare train loss, validation loss, and test loss to diagnose how well the model generalises.

- **Train loss** — how well the model fits the data it was trained on.
- **Validation loss** — how well it generalises to unseen text from the same distribution (monitored during training to detect overfitting early).
- **Test loss** — the final, unbiased estimate of generalisation, computed once on data the model has never influenced.

A large gap between train loss and val/test loss indicates **overfitting**: the model has memorised the training sequences rather than learning general patterns.

In [ ]:
import pandas as pd

# ── Read metrics CSV written by Lightning's CSVLogger ─────────────────────────
df = pd.read_csv(f"{csv_logger.log_dir}/metrics.csv")

train_df = df.dropna(subset=["train_loss"])[["step", "train_loss"]]
val_df   = df.dropna(subset=["val_loss"])[["step", "val_loss"]]

# ── Plot train / val loss curves ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_df["step"], train_df["train_loss"], label="Train loss")
ax.plot(val_df["step"],   val_df["val_loss"],     label="Val loss")
ax.set_xlabel("Step")
ax.set_ylabel("Cross-entropy loss")
ax.set_title("Training and Validation Loss per Step")
ax.legend()
plt.tight_layout()
plt.show()

# ── Test on the best checkpoint (not last epoch) ──────────────────────────────
test_results = trainer.test(ckpt_path="best", datamodule=dm, verbose=False)
print(f"  Test loss (best ckpt): {test_results[0]['test_loss']:.4f}")
# ── Validate on the best checkpoint (not last epoch) ──────────────────────────────
val_results = trainer.validate(ckpt_path="best", datamodule=dm, verbose=False)
print(f"  Val loss (best ckpt): {val_results[0]['val_loss']:.4f}")

---
## Section 6 — Text generation (autoregressive decoding)

### How generation works — step by step

After training, the model has learned the conditional distribution $P(\text{next char} \mid \text{context})$. To generate text we use this distribution iteratively:

1. Start with a short **prompt** encoded as a tensor of token IDs, shape `[1, T]`.
2. Run a **forward pass** → get `logits` of shape `[1, T, vocab_size]`.
3. We only care about the **last position**: `logits[0, -1, :]` is the distribution over the *next* character.
4. Divide by **temperature** (a scalar). Temperature > 1 flattens the distribution (more random); temperature < 1 sharpens it (more greedy).
5. Apply **softmax** to convert logits to probabilities.
6. **Sample** one token from the distribution with `torch.multinomial`. (We could also take the argmax, but sampling produces more varied and creative text.)
7. **Append** the sampled token to the sequence and return to step 2.
8. Repeat for as many tokens as desired.

This is called *autoregressive* decoding because each generated token is fed back as context for the next step. The model's memory is limited to the last `block_size` tokens.

In [ ]:
# ── Autoregressive generation loop ───────────────────────────────────────────
@torch.no_grad()
def generate(
    model: GPTLightning,
    prompt_ids: torch.Tensor,
    max_new_tokens: int,
    block_size: int,
    temperature: float = 1.0,
) -> torch.Tensor:
    """
    Autoregressively generate `max_new_tokens` characters.

    Parameters
    ----------
    model          : trained GPTLightning model
    prompt_ids     : LongTensor of shape [1, T] — the starting context
    max_new_tokens : how many new characters to generate
    block_size     : model's maximum context length
    temperature    : controls randomness (1.0 = unchanged, <1 = more greedy)

    Returns
    -------
    LongTensor of shape [1, T + max_new_tokens]
    """
    model.eval()
    ids = prompt_ids.clone()   # running sequence, grows by 1 each iteration

    for step in range(max_new_tokens):
        # ── Step 1: crop to the last block_size tokens ────────────────────────
        # The model's positional embedding only covers positions 0 … block_size-1.
        # If the sequence is longer, we discard the oldest tokens.
        context = ids[:, -block_size:]

        # ── Step 2: forward pass ──────────────────────────────────────────────
        # We do NOT pass labels here — we want logits, not the loss.
        out    = model.model(input_ids=context)
        logits = out.logits                     # [1, T, vocab_size]

        # ── Step 3: last position only ────────────────────────────────────────
        last_logits = logits[:, -1, :] / temperature   # [1, vocab_size]

        # ── Step 4: softmax → probabilities ──────────────────────────────────
        probs = torch.softmax(last_logits, dim=-1)     # [1, vocab_size]

        # ── Step 5: sample one token ──────────────────────────────────────────
        next_id = torch.multinomial(probs, num_samples=1)   # [1, 1]

        # ── Step 6: append and continue ───────────────────────────────────────
        ids = torch.cat([ids, next_id], dim=1)   # [1, T+1]

    return ids


# ── Load best checkpoint weights before generating ────────────────────────────
_ckpt = torch.load(checkpoint_cb.best_model_path, map_location="cpu")
gpt_pl.load_state_dict(_ckpt["state_dict"])

# ── Generate 500 characters starting from a newline ───────────────────────────
prompt_text = ".\n"
prompt_ids  = torch.tensor([encode(prompt_text)], dtype=torch.long)

generated = generate(
    gpt_pl,
    prompt_ids,
    max_new_tokens = 50,
    block_size     = BLOCK_SIZE,
    temperature    = 1.0,
)

print("─" * 60)
print(decode(generated[0].tolist()))
print("─" * 60)

---
## Section 7 — Attention score visualisation

Each attention head in each transformer block computes a score between every pair of positions. After softmax, these scores tell us **how much each token pays attention to each other token** when building its next representation.

Visualising attention weights as a heatmap is a classic interpretability technique:
- The row axis is the **query** (the current token being processed).
- The column axis is the **key** (the token being looked at).
- A bright cell at `(i, j)` means: when processing position `i`, the model strongly draws information from position `j`.

Because of the causal mask, the upper triangle is always zero — each token can only attend to past tokens.

HF makes extracting attention weights trivial: pass `output_attentions=True` and the model returns a tuple of attention tensors, one per layer.

In [ ]:
# ── Extract and plot attention weights ────────────────────────────────────────
gpt_pl.model.eval()

# Use the first 32 generated characters as the input sequence
sample_text = decode(generated[0, :BLOCK_SIZE].tolist())
sample_ids  = torch.tensor([encode(sample_text)], dtype=torch.long)

with torch.no_grad():
    out = gpt_pl.model(input_ids=sample_ids, output_attentions=True)

# out.attentions is a tuple of length n_layer
# Each element has shape [batch, n_head, seq_len, seq_len]
print(f"Number of layers returned: {len(out.attentions)}")
print(f"Attention tensor shape   : {out.attentions[0].shape}")
print(f"  = [batch={out.attentions[0].shape[0]}, "
      f"n_head={out.attentions[0].shape[1]}, "
      f"seq_len={out.attentions[0].shape[2]}, "
      f"seq_len={out.attentions[0].shape[3]}]")

In [ ]:
# ── Heatmap — change layer_idx / head_idx to explore different heads ──────────
layer_idx = 1   # 0 to N_LAYER-1
head_idx  = 0   # 0 to N_HEAD-1

attn  = out.attentions[layer_idx][0, head_idx].numpy()   # [seq_len, seq_len]
chars = list(sample_text[: attn.shape[0]])

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    attn, ax=ax,
    cmap       = "viridis",
    xticklabels= chars,
    yticklabels= chars,
    linewidths = 0.3,
    linecolor  = "#cccccc",
    vmin=0, vmax=attn.max(),
)
ax.set_title(
    f"Attention weights  —  Layer {layer_idx}, Head {head_idx}\n"
    f"Each row shows where a token allocates its attention (columns sum to 1 within the causal mask)",
    fontsize=11,
)
ax.set_xlabel("Key  (token being attended TO)", fontsize=10)
ax.set_ylabel("Query  (current token)", fontsize=10)
ax.tick_params(axis="both", labelsize=8)
plt.tight_layout()
plt.show()

print()
print("Try changing layer_idx and head_idx above and re-running this cell.")
print("Different heads often specialise: some attend to the previous character,")
print("some to the start of the current word, some to punctuation, etc.")